# Import librerie e dataset

In [1]:
import os

os.chdir("../RecSys_Course_AT_PoliMi")

!pwd


/Users/Filippo/Documents/GitHub/RecSys_Challenge_2025-26/RecSys_Course_AT_PoliMi


In [2]:
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
#%matplotlib inline
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from numpy import linalg as LA
from scipy.sparse import csr_matrix
from sklearn.model_selection import KFold
from skopt.space import Real, Integer, Categorical
from tqdm import tqdm
from xgboost import XGBRanker
import gc
import matplotlib.pyplot as pyplot
import numpy as np
import os
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import scipy.stats as stats
import time 
import tqdm

from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.hybrid.m3_model import TripleIntegratedHierarchicalHybridRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender
from Recommenders.MatrixFactorization.PureSVDRecommender import ScaledPureSVDRecommender
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.XGBoost.feature_populator import feature_populator
from Recommenders.XGBoost.XGBoostRerankerRecommender import XGBoostRerankerRecommender


/Users/Filippo/Documents/GitHub/RecSys_Challenge_2025-26/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [4]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO URM_all
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

# RAM saving
del df_train
gc.collect()

32

In [5]:
best_alpha= 0.15724832635414948
best_beta = 0.08802471282205815
best_gamma = 0.12728641243252908

IALS_Parameters = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595}
 
SLIMElastic_Parameters = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

EASE_R_Parameters = {
    'topK': 2729,
    'l2_norm': 339.5049988216625}


RP3beta_Parameters = {
    'topK': 35,
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'normalize_similarity': True
}

P3alpha_Parameters = {
    'topK': 91,
    'alpha': 0.10032229,
    'normalize_similarity': True
}

ScaledPureSVD_Parameters = {
    'num_factors' : 152,
    'scaling_items' : 0.000714,
    'scaling_users' : 0.575393
}

ItemKNNCF_Parameters = {
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062
}

In [6]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

def optimize_df(df):
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")
        if df[col].dtype == "int64":
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:
models_to_train = [
    (FeatureCombinedImplicitALSRecommender, IALS_Parameters, "IALS"),
    (SLIMElasticNetRecommender, SLIMElastic_Parameters, "SLIMElastic"),
    (EASE_R_Recommender, EASE_R_Parameters, "EASE_R"),
    (RP3betaRecommender, RP3beta_Parameters, "RP3beta"),
    (P3alphaRecommender, P3alpha_Parameters, "P3alpha"),
    (ScaledPureSVDRecommender, ScaledPureSVD_Parameters, "ScaledPureSVD"),
    (ItemKNNCFRecommender, ItemKNNCF_Parameters, "ItemKNNCF")
]

# Creazione di XGBRanker (basato su URM_train)

In [ ]:
# NUOVA CELLA KFOLD

fold_datasets = []
candidates_cutoff = 50

for i in range(5):
    # 1. Definizione Train e Test del fold i
    if i == 4:
        URM_test_fold = URM_parts[i] + URM_parts[0]
        URM_train_fold = sum(URM_parts[j] for j in range(5) if (j != i) & (j != 0))
    else: 
        URM_test_fold = URM_parts[i]+ URM_parts [i+1]
        URM_train_fold = sum(URM_parts[j] for j in range(5) if (j != i) & (j != i+1))  
    
    # 2. Fit dei modelli base
    other_algorithms = {}

    for model_class, params, name in models_to_train:
        print(f"--- Training {name} for fold {i} ---")
        current_recommender = model_class(URM_train_fold)
        current_recommender.fit(**params)

        other_algorithms[name] = current_recommender
        print(f"Modello salvato in: other_algorithms['{name}']\n")

    # Fit del Candidate Generator
    linear_comb_rec = TripleIntegratedHierarchicalHybridRecommender(URM_train_fold, other_algorithms['SLIMElastic'], other_algorithms['EASE_R'], other_algorithms['RP3beta'], other_algorithms['IALS'])
    linear_comb_rec.fit(best_alpha, best_beta, best_gamma)
    
    # 3.1 Creazione del training dataframe per questo specifico fold
    print(f"--- Feature Population for fold {i} ---")
    #training_dataframe = feature_populator(URM_train, linear_comb_rec, other_algorithms, cutoff = candidates_cutoff)
    trainingdf_fold_i = feature_populator(URM_train_fold, linear_comb_rec, other_algorithms, cutoff=candidates_cutoff)
    
    # 3.2 Gestione Label
    URM_test_fold_coo = sps.coo_matrix(URM_test_fold)

    correct_recommendations = pd.DataFrame({"UserID": URM_test_fold_coo.row, "ItemID": URM_test_fold_coo.col})

    trainingdf_fold_i = pd.merge(trainingdf_fold_i, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
    trainingdf_fold_i["Label"] = trainingdf_fold_i["Exist"] == "both"
    trainingdf_fold_i.drop(columns = ['Exist'], inplace=True)
    
    # 4. X e y per questo fold

    # questa lista serve ad avere sempre le stesse feature nello stesso ordine per ogni fold
    
    groups_fold = trainingdf_fold_i.groupby("UserID").size().values

    # 2. Creazione X e y
    y_train = trainingdf_fold_i["Label"]
    X_train = trainingdf_fold_i.drop(columns=["Label"])

    # Conversione tipi (OK come int, ma attenzione se sono univoci)
    X_train["UserID"] = X_train["UserID"].astype(int)
    X_train["ItemID"] = X_train["ItemID"].astype(int)
    
    fold_datasets.append({
        "X": X_train, 
        "y": y_train, 
        "groups": groups_fold,
        "full_df": trainingdf_fold_i,
        "URM_test": URM_test_fold 
    })
    
    print(f"Fold {i} completato.")

    del other_algorithms
    del linear_comb_rec
    del trainingdf_fold_i
    del correct_recommendations
    gc.collect()

--- Training IALS for fold 0 ---
Modello salvato in: other_algorithms['IALS']

--- Training SLIMElastic for fold 0 ---


# Training hybrid su URM_train_validation

In [ ]:
# NUOVA CELLA KFOLD

fold_datasets_val = []
candidates_cutoff = 50

for i in range(5):
    # 1. Definizione Train e Test del fold i

    URM_test_fold = URM_parts[i] 
    URM_train_fold = sum(URM_parts[j] for j in range(5) if j != i)
   
    # 2. Fit dei modelli base
    other_algorithms = {}

    for model_class, params, name in models_to_train:
        print(f"--- Training {name} for fold {i} ---")
        current_recommender = model_class(URM_train_fold)
        current_recommender.fit(**params)

        other_algorithms[name] = current_recommender
        print(f"Modello salvato in: other_algorithms['{name}']\n")

    # Fit del Candidate Generator
    linear_comb_rec_f = TripleIntegratedHierarchicalHybridRecommender(URM_train_fold, other_algorithms['SLIMElastic'], other_algorithms['EASE_R'], other_algorithms['RP3beta'], other_algorithms['IALS'])
    linear_comb_rec_f.fit(best_alpha, best_beta, best_gamma)
    
    # 3.1 Creazione del training dataframe per questo specifico fold
    print(f"--- Feature Population for fold {i} ---")
    #training_dataframe = feature_populator(URM_train, linear_comb_rec, other_algorithms, cutoff = candidates_cutoff)
    trainingdf_fold_i = feature_populator(URM_train_fold, linear_comb_rec_f, other_algorithms, cutoff=candidates_cutoff)
    
    trainingdf_fold_i["UserID"] = trainingdf_fold_i["UserID"].astype(int)
    trainingdf_fold_i["ItemID"] = trainingdf_fold_i["ItemID"].astype(int)

    
    fold_datasets_val.append({ 
        "URM_train_validation": URM_train_fold,
        "full_df": trainingdf_fold_i,
        "URM_test": URM_test_fold 
    })
    
    print(f"Fold {i} completato.")

    del other_algorithms
    del trainingdf_fold_i
    gc.collect()

--- Training IALS for fold 0 ---
Modello salvato in: other_algorithms['IALS']

--- Training SLIMElastic for fold 0 ---
SLIMElasticNetRecommender: Processed 4805 (68.9%) in 5.00 min. Items per second: 16.01
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.27 min. Items per second: 15.98
Modello salvato in: other_algorithms['SLIMElastic']

--- Training EASE_R for fold 0 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 12.46 sec
Modello salvato in: other_algorithms['EASE_R']

--- Training RP3beta for fold 0 ---
RP3betaRecommender: Similarity column 6969 (100.0%), 3452.41 column/sec. Elapsed time 2.02 sec
Modello salvato in: other_algorithms['RP3beta']

--- Training P3alpha for fold 0 ---
P3alphaRecommender: Similarity column 6969 (100.0%), 3300.68 column/sec. Elapsed time 2.11 sec
Modello salvato in: other_algorithms['P3alpha']

--- Training ScaledPureSVD for fold 0 ---
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecom

100%|██████████| 27095/27095 [00:17<00:00, 1536.55it/s]


---------------3---------------


100%|██████████| 7/7 [01:39<00:00, 14.21s/it]


--------------- 4 ---------------
Processing SLIMElastic...


...

Users RP3beta: 100%|██████████| 27095/27095 [00:35<00:00, 756.05it/s]


Processing ItemKNNCF...


Users ItemKNNCF: 100%|██████████| 27095/27095 [00:36<00:00, 748.17it/s]


---------------6---------------
---------------7---------------
--------------- 5 ---------------
--------------- ScaledPureSVD Embeddings ---------------
--------------- Score Ratios & Rank Differences ---------------
training dataframe creato con successo
Fold 1 completato.
--- Training IALS for fold 2 ---
Modello salvato in: other_algorithms['IALS']

--- Training SLIMElastic for fold 2 ---


KeyboardInterrupt: 

# Hyper-parameters tuning con optuna

In [ ]:
import optuna
from xgboost import XGBRanker

In [ ]:
class XGBoostRerankerRecommender:
    def __init__(self, URM_train, XGB_model, df):
        self.URM_train = URM_train
        self.df = df
        self.XGB_model = XGB_model

    def recommend(self, user_ids, cutoff=20, return_scores=True, remove_seen_flag=True, remove_top_pop_flag=True, remove_custom_items_flag=False):
        recommendations = []
        for user_id in user_ids:
            # print(user_id)
            df_slice = self.df[self.df['UserID'] == user_id]
            items = df_slice.ItemID.to_numpy()
            preds = self.XGB_model.predict(df_slice)
            recommendations.append(items[np.argsort(preds)[-cutoff:][::-1]].tolist())
        
        if return_scores:
            rec, scores = linear_comb_rec_f.recommend(user_ids, cutoff=cutoff, return_scores=return_scores)
            # useless scores
            return np.array(recommendations), scores
        
        return np.array(recommendations)

    def get_URM_train(self):
        return self.URM_train

In [ ]:
def objective_xgboost_kfold(trial):
    
    start_time = time.time()
    scores = []
    
    for i in range (5):
        
        XGB_model = XGBRanker(
            objective = 'rank:pairwise',
            n_estimators = trial.suggest_int('n_estimators', 1000, 3500, log=True),
            learning_rate = trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
            reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 1, log=True),
            reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 5, log=True),
            max_depth = trial.suggest_int('max_depth', 3, 8),
            max_leaves = trial.suggest_int('max_leaves', 32, 512),
            grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
            verbosity = 2,
            booster = 'gbtree',
            tree_method = 'hist',
            gamma = trial.suggest_float('gamma', 1e-2, 10, log=True),
            min_child_weight = trial.suggest_float('min_child_weight', 0.01, 10, log=True),
            subsample = trial.suggest_float('subsample', 0.5, 0.9),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 0.9),
            # enable_categorical = True
        )

        XGB_model.fit(
            fold_datasets[i]["X"],
            fold_datasets[i]["Y"],
            group=fold_datasets[i]["groups"],
            verbose=True
        )

        recommender = XGBoostRerankerRecommender(fold_datasets_val[i]['URM_train_validation'], XGB_model, fold_datasets_val[i]['full_df'])
        
        evaluator = EvaluatorHoldout(fold_datasets_val[i]['URM_test'], cutoff_list=[20])
        result_df, _ = evaluator.evaluateRecommender(recommender)
        scores.append(result_df["RECALL"].values[0])
        
        del recommender
        gc.collect()

    return sum(scores) / len(scores)

In [ ]:
study = optuna.create_study(direction='maximize', study_name='xgboost_kfold')
study.optimize(objective_xgboost_kfold, n_trials=100)

# Parametri finali xgboost

In [ ]:
#[I 2025-12-23 15:40:49,840] Trial 22 finished with value: 0.29571209849373575 and parameters: {'n_estimators': 1930, 'learning_rate': 0.00793921492462949, 'reg_alpha': 0.3593999927562866, 'reg_lambda': 2.8188954368881356e-05, 'max_depth': 6, 'max_leaves': 771, 'grow_policy': 'depthwise', 'gamma': 0.8481231854795428, 'min_child_weight': 0.08724326447453803, 'subsample': 0.4747676555813949, 'colsample_bytree': 0.5793024464609432}. Best is trial 22 with value: 0.29571209849373575.

In [ ]:
# [I 2025-12-22 17:54:56,344] Trial 4 finished with value: 0.2953780110258214 and parameters: {'n_estimators': 3542, 'learning_rate': 0.005151155715154754, 'reg_alpha': 6.74624297235953e-05, 'reg_lambda': 1.8028499179599417e-05, 'max_depth': 5, 'max_leaves': 824, 'grow_policy': 'depthwise', 'gamma': 0.22904628185587037, 'min_child_weight': 1.173447891823432, 'subsample': 0.38863812901584477, 'colsample_bytree': 0.658249199203611}. Best is trial 4 with value: 0.2953780110258214.

In [ ]:
#[I 2025-12-31 22:28:07,701] Trial 65 finished with value: 0.2948265841354942 and parameters: {'n_estimators': 4701, 'learning_rate': 0.005902728181292574, 'reg_alpha': 0.014622474581944843, 'reg_lambda': 0.0022623497938618086, 'max_depth': 7, 'max_leaves': 325, 'grow_policy': 'lossguide', 'gamma': 1.4057752317825953, 'min_child_weight': 8.462525277265964e-07, 'subsample': 0.7498497263226135, 'colsample_bytree': 0.38978479584705306}. Best is trial 65 with value: 0.2948265841354942.

In [ ]:
#[I 2026-01-02 08:30:05,705] Trial 96 finished with value: 0.2958264257536273 and parameters: {'n_estimators': 2261, 'learning_rate': 0.007702162822426045, 'reg_alpha': 0.00991022085129717, 'reg_lambda': 0.00014710271979317914, 'max_depth': 7, 'max_leaves': 349, 'grow_policy': 'lossguide', 'gamma': 0.3793589358094316, 'min_child_weight': 0.002662290661537342, 'subsample': 0.6587292706752251, 'colsample_bytree': 0.48594318797001984}. Best is trial 96 with value: 0.2958264257536273.

In [ ]:
XGBoost_Parameters = {    
    'objective': 'rank:map', 
    'n_estimators': 2261, 
    'learning_rate': 0.007702162822426045, 
    'reg_alpha': 0.00991022085129717, 
    'reg_lambda': 0.00014710271979317914, 
    'max_depth': 7, 
    'max_leaves': 349, 
    'grow_policy': 'lossguide', 
    'gamma': 0.3793589358094316, 
    'min_child_weight': 0.002662290661537342,
    'subsample': 0.6587292706752251, 
    'colsample_bytree': 0.48594318797001984
}

In [ ]:
# RAM Saving

del training_dataframe
del validation_dataframe
del X_train
del y_train
gc.collect() 

# Creazione final_train_dataframe

In [ ]:
final_train_dataframe = feature_populator(URM_train_validation, linear_comb_rec_f, other_algorithms_f, cutoff = candidates_cutoff)

# RAM Saving
final_train_dataframe = optimize_df(final_train_dataframe)

In [ ]:
URM_test_coo = sps.coo_matrix(URM_test)

correct_recommendations_all = pd.DataFrame({"UserID": URM_test_coo.row,
                                        "ItemID": URM_test_coo.col})
correct_recommendations_all

In [ ]:
final_train_dataframe = pd.merge(final_train_dataframe, correct_recommendations_all, on=['UserID','ItemID'], how='left', indicator='Exist')
final_train_dataframe["Label"] = final_train_dataframe["Exist"] == "both"
final_train_dataframe.drop(columns = ['Exist'], inplace=True)
final_train_dataframe

# Training finale XGBoost

In [ ]:

final_train_dataframe

In [ ]:
groups = final_train_dataframe.groupby("UserID").size().values
groups

In [ ]:
from xgboost import XGBRanker
XGB_model_f = XGBRanker(**XGBoost_Parameters)
#XGB_model_f = XGBRanker()
y_train = final_train_dataframe["Label"]
X_train = final_train_dataframe.drop(columns=["Label"])
X_train["UserID"] = X_train["UserID"].astype(int)
X_train["ItemID"] = X_train["ItemID"].astype(int)

#print(np.isinf(X_train).sum())  # Count of infinite values in training data


# Check for NaN values
#print(np.isnan(X_train).sum())  # Count of NaN values in training data

#X_train = np.nan_to_num(X_train, posinf=np.finfo(np.float32).max, neginf=0.0)

XGB_model_f.fit(
    X_train,
    y_train,
    group=groups,
    verbose=True
)

In [ ]:
#%matplotlib inline
from xgboost import plot_importance

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 25))
plot_importance(XGB_model_f, importance_type='weight', title='Weight (Frequence)', max_num_features=100, ax=ax)
#plt.show()
plt.savefig('feature_importance_03012026.png')

import pandas as pd
try:
    importance_dict = XGB_model_f.get_score(importance_type='weight')
except AttributeError:
    importance_dict = XGB_model_f.get_booster().get_score(importance_type='weight')
importance_df = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importance': list(importance_dict.values())
}).sort_values(by='Importance', ascending=False)
with pd.option_context('display.max_rows', None):
    print(importance_df)

In [ ]:
# RAM Saving
import gc

del other_algorithms
del other_algorithms_f
del linear_comb_rec
del linear_comb_rec_f
del training_dataframe
del validation_dataframe

gc.collect()

# Training modelli con URM_all

In [ ]:
other_algorithms_all = {}

# Loop per trainare e valutare tutti i modelli
for model_class, params, name in models_to_train:
    print(f"--- Training {name} ---")
    
    current_recommender = model_class(URM_all)
    current_recommender.fit(**params)
    
    other_algorithms_all[name] = current_recommender
    
    print(f"Modello salvato in: other_algorithms_all['{name}']\n")

other_algorithms_all

In [ ]:
linear_comb_rec_all = TripleIntegratedHierarchicalHybridRecommender(URM_all, other_algorithms_all['SLIMElastic'], other_algorithms_all['EASE_R'], other_algorithms_all['RP3beta'], other_algorithms_all['IALS'])
linear_comb_rec_all.fit(best_alpha, best_beta, best_gamma)

# Creazione prediction_dataframe

In [ ]:
prediction_dataframe = feature_populator(URM_all, linear_comb_rec_all, other_algorithms_all, cutoff = candidates_cutoff)

# RAM Saving
#prediction_dataframe = optimize_df(prediction_dataframe)

prediction_dataframe

# Run finale XGBoostRerankerRecommender + submission

In [ ]:
prediction_dataframe["UserID"] = prediction_dataframe["UserID"].astype(int)
prediction_dataframe["ItemID"] = prediction_dataframe["ItemID"].astype(int)

In [ ]:
recommender = XGBoostRerankerRecommender(URM_all, XGB_model_f, prediction_dataframe)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = recommender.recommend([user_id], cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations[0][0]))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_xgboost_02012026.csv", index=False)

end_time = time.time()